# Build a Small Language Model from Scratch — on your own stories

This follows the Vizuara notebook step for step. **Step 4, the model
architecture, is copied from it unchanged** — that is the cell the video
explains, and nothing here modifies it.

What is different is everything *around* the model, because the dataset is
different. TinyStories is 2.1M simple children's stories; yours is ~2.4M
narrated Reddit-drama stories of 230–280 words with a deliberate beat
structure. Six changes follow from that, and each one is marked `# CHANGED:`
in the cell where it happens:

| | video | here | why |
|---|---|---|---|
| tokenizer | GPT-2, 50,257 | custom, 8,192 | 50k vocab = 19.3M params of embedding table, more than your whole model budget |
| `block_size` | 128 | 512 | your stories are ~350 tokens; at 128 the model never sees a hook and its payoff together |
| story separator | none | `<\|endoftext\|>` | lets you generate a *fresh* story instead of priming with text |
| validation split | random | by template | 357 stories share each template, so a random split measures recall, not learning |
| dropout | 0.1 | 0.0 | you have ~60x more tokens than parameters; the model cannot memorise the corpus |
| checkpoints | model only | full state | Colab/Kaggle will disconnect mid-run; this resumes |

Four bugs in the original are also fixed, marked `# BUGFIX:`. The largest:
`min_lr` was set *above* `learning_rate`, so the learning rate rose through
training instead of decaying.

Run the cells in order.

## Step 0: Settings — the only cell you need to edit

In [ ]:
import os, glob

# ---------------------------------------------------------------------------
# WHERE YOUR STORIES ARE.
# Point this at your JSONL. It reads BOTH shapes automatically:
#   * cleaned rows  -> {"story_text": ...}          (clean_dataset.py output)
#   * raw rows      -> {"request":..., "response":...}  (Vertex batch output)
# so you can start before the cleaning pass has finished.
# ---------------------------------------------------------------------------
DATA_GLOB = "data/batch_results/**/*.jsonl"

# Colab:  mount Drive, then use something like
#   from google.colab import drive; drive.mount('/content/drive')
#   DATA_GLOB = "/content/drive/MyDrive/yt-ai/batch_results/**/*.jsonl"
# Kaggle: add your dataset, then
#   DATA_GLOB = "/kaggle/input/<your-dataset>/**/*.jsonl"

WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
OUT_DIR  = os.path.join(WORK_DIR, "slm_out")
os.makedirs(OUT_DIR, exist_ok=True)

# ---- model size -----------------------------------------------------------
VOCAB_SIZE = 8192      # CHANGED: was 50257 (GPT-2). See the table above.
BLOCK_SIZE = 512       # CHANGED: was 128.
DROPOUT    = 0.0       # CHANGED: was 0.1.

# The video's config reports 30.0M from torch; count the tied embedding twice
# and you get the 49.3M often quoted. Either way only 10.6M of it is
# transformer -- the rest is the GPT-2 vocabulary lookup table.
#
# These presets hit the same headline numbers with an 8k vocabulary, so the
# parameters go into depth and width instead. "50m" matches that 49.3M almost
# exactly while carrying 4.2x more transformer.
SIZES = {
    "15m": dict(n_layer=6,  n_head=6,  n_embd=384),   # 13.8M,  10.6M transformer
    "30m": dict(n_layer=8,  n_head=8,  n_embd=512),   # 29.6M,  25.2M transformer
    "50m": dict(n_layer=9,  n_head=10, n_embd=640),   # 49.8M,  44.2M transformer
    "60m": dict(n_layer=8,  n_head=12, n_embd=768),   # 63.3M,  56.6M transformer
}
MODEL_SIZE = "50m"
N_LAYER = SIZES[MODEL_SIZE]["n_layer"]
N_HEAD  = SIZES[MODEL_SIZE]["n_head"]
N_EMBD  = SIZES[MODEL_SIZE]["n_embd"]

# ---- how long to train ----------------------------------------------------
EPOCHS    = 2.0        # passes over your token stream
MAX_HOURS = 11.0       # checkpoint + stop before the platform kills you.
                       # Kaggle session limit is 12h, Colab free is ~4h.

# ---- data prep ------------------------------------------------------------
MAX_STORIES    = 0     # 0 = all. Set e.g. 200000 for a fast first run.
STRIP_ITALICS  = True  # raw data contains *single-word italics*
MIN_WORDS, MAX_WORDS = 150, 450
VAL_TEMPLATE_PERMILLE = 20   # 2% of TEMPLATES held out entirely

USE_GPT2_TOKENIZER = False   # emergency fallback: True = tiktoken, 50257 vocab
                             # and a ~30M model. Only if the custom one fails.

files = sorted(glob.glob(DATA_GLOB, recursive=True))
print(f"{len(files)} file(s) matched {DATA_GLOB}")
if not files:
    raise SystemExit("No files matched. Fix DATA_GLOB before continuing.")
print("first:", files[0])

In [ ]:
!pip install -q tokenizers tiktoken

### Getting your data into Colab

Do **not** upload the JSONL. Run Steps 1-2 once on your Mac (both are CPU-only)
and you turn ~4 GB of JSONL into ~1.7 GB of `train.bin` plus a 400 KB
`tokenizer.json`. Upload *those*. Re-tokenising 2.4M stories at the start of
every session burns GPU time you are paying for.

A private Hugging Face dataset repo is the best home for them: the download is
parallel and resumable, there is no Drive quota, and it survives every runtime
reset. Google Drive also works and needs no account setup, but Colab's Drive
mount is slow and flaky on multi-GB files.

Use **the dataset version that still has `source_template_id` or the `id`
field.** The story-text-only export cannot be split by template, and that split
is the only honest validation signal you have.

Uncomment whichever applies.

In [ ]:
# --- Option A: private Hugging Face dataset (recommended) ------------------
# from huggingface_hub import snapshot_download
# from getpass import getpass
# snapshot_download(repo_id="your-username/yt-ai-stories", repo_type="dataset",
#                   local_dir=OUT_DIR, token=getpass("HF token: "))
# # then skip Steps 1-2: the .bin files are already there.

# --- Option B: Google Drive ------------------------------------------------
# from google.colab import drive; drive.mount("/content/drive")
# DATA_GLOB = "/content/drive/MyDrive/yt-ai/stories.jsonl"

# --- Uploading FROM your Mac (run locally, not here) -----------------------
#   pip install huggingface_hub
#   huggingface-cli login
#   huggingface-cli upload your-username/yt-ai-stories ./slm_out \
#       --repo-type dataset --private
print("configured:", OUT_DIR)

## Step 1: Import the dataset

The video calls `load_dataset("roneneldan/TinyStories")`. Your stories are
local JSONL, so this reads them directly — no `datasets` dependency.

**CHANGED: the train/validation split is by TEMPLATE, not by story.** Each of
your ~7,270 templates backs ~357 stories that share a beat sequence, an
emotional arc and a narrative voice. Split at random and every held-out story
has 356 near-siblings in training, so validation loss measures recall — it
will look excellent while telling you nothing. Hashing the template id keeps
whole templates on one side. `extra_` run prefixes are stripped first so a
template cannot straddle the split.

In [ ]:
import json, hashlib, re, gzip
from collections import Counter

_ITALIC = re.compile(r"(?<![*\w])\*(?=[^\s*])([^*\n]{1,60}?)(?<=[^\s*])\*(?![*\w])")
_ID     = re.compile(r"^(?:[a-z]+_)?(.*)_v\d+$")

def find_text(obj):
    """First string-valued 'text' anywhere in a nested structure."""
    if isinstance(obj, dict):
        if isinstance(obj.get("text"), str):
            return obj["text"]
        for v in obj.values():
            f = find_text(v)
            if f: return f
    elif isinstance(obj, list):
        for it in obj:
            f = find_text(it)
            if f: return f
    return None

def extract(row):
    rid = row.get("id") or row.get("custom_id")
    if isinstance(row.get("story_text"), str):          # cleaned shape
        return rid, row["story_text"]
    for key in ("response", "predictions", "prediction", "candidates"):
        if key in row:                                   # raw shape
            # Search the RESPONSE only. The echoed request also has a "text"
            # field, and picking that up trains the model on its own prompts.
            return rid, find_text(row[key])
    return rid, find_text({k: v for k, v in row.items() if k != "request"})

def template_of(row, rid):
    t = row.get("source_template_id")
    if t: return str(t)
    m = _ID.match(rid or "")
    return m.group(1) if m else (rid or "")

def bucket(key, mod):
    return int(hashlib.md5(key.encode()).hexdigest()[:8], 16) % mod

stats, splits = Counter(), {"train": [], "val_new": [], "val_seen": []}
tpl_seen = {"train": set(), "val_new": set()}

for path in files:
    op = gzip.open if path.endswith(".gz") else open
    with op(path, "rt", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            stats["read"] += 1
            try: row = json.loads(line)
            except json.JSONDecodeError:
                stats["bad_json"] += 1; continue

            rid, text = extract(row)
            if not text or not text.strip():
                stats["no_text"] += 1; continue
            text = text.strip()
            if STRIP_ITALICS:
                text = _ITALIC.sub(r"\1", text)

            n = len(text.split())
            if n < MIN_WORDS: stats["too_short"] += 1; continue
            if n > MAX_WORDS: stats["too_long"]  += 1; continue

            tpl = template_of(row, rid)
            if bucket(tpl, 1000) < VAL_TEMPLATE_PERMILLE:
                split = "val_new"
            elif rid and bucket(rid, 400) == 0:
                split = "val_seen"          # held-out stories from TRAIN templates
            else:
                split = "train"
            splits[split].append(text)
            if split in tpl_seen: tpl_seen[split].add(tpl)
            stats["kept"] += 1
            if MAX_STORIES and stats["kept"] >= MAX_STORIES: break
    if MAX_STORIES and stats["kept"] >= MAX_STORIES: break

for k, v in stats.items(): print(f"  {k:12s} {v:>10,}")
print()
for s in ("train", "val_new", "val_seen"):
    print(f"  {s:9s} {len(splits[s]):>9,} stories")
_bytes = sum(len(t) for v in splits.values() for t in v)
print(f"  ~{_bytes/1e9:.2f} GB of story text held in RAM "
      f"(roughly {_bytes*1.6/1e9:.1f} GB as Python strings)")
if _bytes * 1.6 > 8e9:
    print("  WARNING: that is a lot for Colab free (~12.7 GB total). If")
    print("  this cell or the next one dies, set MAX_STORIES (e.g.")
    print("  1000000) and re-run, or use Kaggle, which has ~30 GB.")
print(f"\n  templates: {len(tpl_seen['train']):,} train / "
      f"{len(tpl_seen['val_new']):,} val")
leak = tpl_seen["train"] & tpl_seen["val_new"]
print(f"  templates in BOTH (must be 0): {len(leak)}")
assert not leak, "templates straddle the split -- validation is contaminated"
assert splits["train"], "no training stories -- check DATA_GLOB and the word bounds"
if not splits["val_new"]:
    print("=" * 70)
    print("val_new is EMPTY: no template hashed into the held-out bucket.")
    print("It is the only number here that measures generalisation, so do not")
    print("train without it. Raise VAL_TEMPLATE_PERMILLE (currently "
          f"{VAL_TEMPLATE_PERMILLE}), or check that your ids look like")
    print("'<template>_v<n>' so the template can be parsed out of them.")
    print("=" * 70)
    raise SystemExit("empty validation split")


## Step 2: Tokenize the dataset

The video uses `tiktoken.get_encoding("gpt2")` — 50,257 tokens. At
`n_embd=384` that embedding table alone is `50257 x 384 = 19.3M` parameters,
which is **larger than the entire model you are trying to build**, and it
makes the video's model ~30M rather than ~15M.

**CHANGED: train an 8,192-token vocabulary on your own stories instead.**
That costs 3.15M parameters, and because the corpus is one narrow domain the
merges fit it better than GPT-2's general-purpose ones. Byte-level, so curly
quotes and em dashes can never produce an unknown token.

This cell takes a few minutes on the full corpus. Nothing else depends on how
long it takes.

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

EOT = "<|endoftext|>"
TOK_PATH = os.path.join(OUT_DIR, "tokenizer.json")

if USE_GPT2_TOKENIZER:
    import tiktoken
    _enc = tiktoken.get_encoding("gpt2")
    encode = lambda s: _enc.encode_ordinary(s)
    decode = lambda ids: _enc.decode(ids)
    eot_id, vocab_size = _enc.eot_token, 50257
else:
    if not os.path.exists(TOK_PATH):
        tok = Tokenizer(models.BPE(unk_token=None))
        tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
        tok.decoder = decoders.ByteLevel()
        trainer = trainers.BpeTrainer(
            vocab_size=VOCAB_SIZE, min_frequency=2,
            special_tokens=[EOT],                 # id 0
            initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
            show_progress=True)
        # Fit on TRAIN only. A tokenizer fitted on validation text is a real
        # leak: merges tuned to held-out strings make them cheaper to predict.
        stride = max(1, len(splits["train"]) // 300_000)
        tok.train_from_iterator(splits["train"][::stride], trainer=trainer)
        tok.save(TOK_PATH)
    tok = Tokenizer.from_file(TOK_PATH)
    encode = lambda s: tok.encode(s).ids
    decode = lambda ids: tok.decode(ids)
    eot_id, vocab_size = tok.token_to_id(EOT), tok.get_vocab_size()

print(f"vocab={vocab_size:,}  {EOT}={eot_id}")

# The measurement that sets your compute budget.
_s = splits["train"][:2000]
_w = sum(len(t.split()) for t in _s)
_t = sum(len(encode(t)) + 1 for t in _s)
print(f"{_t/_w:.3f} tokens/word, {_t/len(_s):.0f} tokens/story")
print(f"embedding table = {vocab_size*N_EMBD/1e6:.2f}M parameters")
print("round-trip lossless:", decode(encode(_s[0])) == _s[0])

### Write `train.bin` / `val.bin`

Same idea as the video: concatenate every story into one flat `uint16` array
and memory-map it.

**CHANGED: `<|endoftext|>` is appended after each story.** The video's
`process()` drops nanoGPT's `ids.append(enc.eot_token)`, so its stories run
together with no boundary — which is why its inference has to be primed with
`"Once upon a time..."`. With the separator, the model learns where a story
starts and ends, and you can ask it for a fresh one.

In [ ]:
import numpy as np
from tqdm.auto import tqdm

def write_bin(texts, name):
    path = os.path.join(OUT_DIR, f"{name}.bin")
    if os.path.exists(path) and os.path.getsize(path) > 0:
        n = os.path.getsize(path) // 2
        print(f"  {name}.bin exists: {n:,} tokens"); return n
    total = 0
    with open(path, "wb") as f:
        for i in tqdm(range(0, len(texts), 2000), desc=f"writing {name}.bin"):
            ids = []
            for t in texts[i:i+2000]:
                ids.extend(encode(t))
                ids.append(eot_id)          # CHANGED: story boundary
            a = np.asarray(ids, dtype=np.uint16)
            assert not a.size or int(a.max()) < 65536
            a.tofile(f); total += a.size
    print(f"  {name}.bin: {total:,} tokens ({os.path.getsize(path)/1e9:.2f} GB)")
    return total

train_tokens = write_bin(splits["train"],   "train")
val_tokens   = write_bin(splits["val_new"], "val_new")
seen_tokens  = write_bin(splits["val_seen"],"val_seen")

# The .bin files are the training data from here on; the Python lists are
# several GB of dead weight. Freeing them is the difference between
# fitting in Colab's RAM and not.
import gc
splits = {k: v[:3] for k, v in splits.items()}   # keep a few to inspect
gc.collect()

print(f"\nOne pass over train = {train_tokens:,} tokens.")

## Step 3: Create input–output batches

Unchanged from the video (which takes it from nanoGPT). Random `block_size`
windows out of the packed stream; the memmap is reopened every call because
holding one open across thousands of iterations leaks memory.

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
device_type = "cuda" if "cuda" in device else "cpu"

_paths = {"train": os.path.join(OUT_DIR, "train.bin"),
          "val_new": os.path.join(OUT_DIR, "val_new.bin"),
          "val_seen": os.path.join(OUT_DIR, "val_seen.bin")}

def get_batch(split):
    data = np.memmap(_paths[split], dtype=np.uint16, mode="r")
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])
    if device_type == "cuda":
        x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)
    return x, y

print("device:", device)

## Step 4: Define the SLM model architecture

**This cell is copied from the Vizuara notebook unchanged.** It is the
architecture the video explains: LayerNorm, causal self-attention, a 4x GELU
MLP, learned position embeddings, weight tying between the input embedding and
the output head.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass
import numpy as np
from tqdm.auto import tqdm
from contextlib import nullcontext
import os

class LayerNorm(nn.Module):
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.flash = hasattr(F, 'scaled_dot_product_attention')
        if not self.flash:
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                       .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        if self.flash:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.attn_dropout.p if self.training else 0.0, is_causal=True)
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)
    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = LayerNorm(config.n_embd, config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln2 = LayerNorm(config.n_embd, config.bias)
        self.mlp = MLP(config)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

@dataclass
class GPTConfig:
    block_size: int
    vocab_size: int
    n_layer: int
    n_head: int
    n_embd: int
    dropout: float = 0.0
    bias: bool = True

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),
            wpe=nn.Embedding(config.block_size, config.n_embd),
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f=LayerNorm(config.n_embd, config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight  # weight tying

        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size
        pos = torch.arange(0, t, dtype=torch.long, device=device)

        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
            return logits, loss
        else:
            logits = self.lm_head(x[:, [-1], :])
            return logits, None

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        Generate tokens given a conditioning sequence.
        idx: Tensor of shape (B, T)
        """
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx



In [ ]:
config = GPTConfig(
    vocab_size=vocab_size,   # CHANGED: 8192 from your tokenizer, not 50257
    block_size=BLOCK_SIZE,   # CHANGED: 512, not 128
    n_layer=N_LAYER,
    n_head=N_HEAD,
    n_embd=N_EMBD,
    dropout=DROPOUT,         # CHANGED: 0.0, not 0.1
    bias=True,
)
model = GPT(config).to(device)

n_params = sum(p.numel() for p in model.parameters())
n_emb    = model.transformer.wte.weight.numel() + model.transformer.wpe.weight.numel()
print(f"{n_params/1e6:.2f}M parameters "
      f"({n_emb/1e6:.2f}M embeddings = {n_emb/n_params*100:.0f}%, "
      f"{(n_params-n_emb)/1e6:.2f}M transformer)")
print(f"With the GPT-2 vocabulary this same model would be "
      f"{(n_params - n_emb + 50257*N_EMBD + BLOCK_SIZE*N_EMBD)/1e6:.1f}M.")

## Step 5: Define the loss function

**BUGFIX.** The video uses `eval_iters` as *both* the evaluation interval and
the number of evaluation batches, so with `eval_iters=500` and
`max_iters=20000` it runs 40 evaluations x 500 batches x 2 splits = 40,000
evaluation passes against 20,000 training steps — more compute spent measuring
than learning. They are separate settings here.

Two validation losses are tracked. **The gap between them is your
memorisation signal**: `val_seen` (held-out stories from templates the model
trained on) minus `val_new` (templates it has never seen). Near zero means it
learned English narrative prose. Growing means it learned 7,270 beat skeletons
and is reciting them.

In [ ]:
@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ("train", "val_new", "val_seen"):
        if not os.path.exists(_paths[split]) or os.path.getsize(_paths[split]) == 0:
            continue
        losses = torch.zeros(EVAL_BATCHES)
        for k in range(EVAL_BATCHES):
            X, Y = get_batch(split)
            with ctx:
                _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

## Step 6: Training configuration

**BUGFIX: `min_lr` must be BELOW `learning_rate`.** The video sets
`learning_rate=1e-4` and `min_lr=5e-4`. `CosineAnnealingLR(eta_min=5e-4)`
then inverts — I traced the actual schedule and it climbs 1e-4 → 5e-4
monotonically across all 20,000 iterations instead of decaying.

`max_iters` is computed from `EPOCHS` rather than hard-coded. The video's
20,000 x 32 x 128 = 82M tokens is 0.17 of one pass over TinyStories; on your
corpus that fraction would be far smaller still.

In [ ]:
from contextlib import nullcontext

batch_size = 32
block_size = BLOCK_SIZE
gradient_accumulation_steps = 4      # 32 x 4 x 512 = 65,536 tokens per update

learning_rate = 1e-3                 # peak; small models tolerate this
min_lr        = 1e-4                 # BUGFIX: below the peak, so cosine DECAYS
grad_clip     = 1.0
EVAL_BATCHES  = 50                   # BUGFIX: separate from the interval

tokens_per_update = batch_size * gradient_accumulation_steps * block_size
total_updates = max(1, int(EPOCHS * train_tokens / tokens_per_update))
max_iters     = total_updates * gradient_accumulation_steps

# Scale warmup and the eval interval to the run length. Fixed values of 500
# silently break a short first run: warmup longer than the run means the
# learning rate never decays, and an eval interval longer than the run means
# no evaluation ever fires, so best_model_params.pt is never written and
# Step 10 fails to load.
warmup_steps  = min(500, max(1, total_updates // 20))
EVAL_INTERVAL = min(500, max(1, total_updates // 10))

dtype = ("bfloat16" if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
         else "float16" if torch.cuda.is_available() else "float32")
ptdtype = {"float32": torch.float32, "bfloat16": torch.bfloat16, "float16": torch.float16}[dtype]
ctx = nullcontext() if device_type == "cpu" else torch.amp.autocast(device_type=device_type, dtype=ptdtype)

torch.manual_seed(42)

print(f"dtype={dtype}")
print(f"{tokens_per_update:,} tokens/update, {total_updates:,} updates "
      f"({EPOCHS} epochs over {train_tokens:,} tokens)")
print(f"warmup {warmup_steps:,} updates, evaluating every {EVAL_INTERVAL:,}")

## Step 7: Optimizer and scheduler

**BUGFIX: the scheduler steps once per *optimizer* update, not once per
micro-batch.** The video calls `scheduler.step()` every iteration while the
optimizer only steps every 32, so its "1000-step warmup" is really about 31
weight updates.

In [ ]:
from torch.optim.lr_scheduler import LinearLR, SequentialLR, CosineAnnealingLR

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate,
                              betas=(0.9, 0.95), weight_decay=0.1, eps=1e-9)
scheduler = SequentialLR(
    optimizer,
    schedulers=[LinearLR(optimizer, total_iters=warmup_steps),
                CosineAnnealingLR(optimizer, T_max=max(total_updates - warmup_steps, 1),
                                  eta_min=min_lr)],
    milestones=[warmup_steps])
scaler = torch.amp.GradScaler(device_type, enabled=(dtype == "float16"))

# Confirm the schedule actually decays before spending hours on it.
_probe = []
for _ in range(total_updates):
    _probe.append(optimizer.param_groups[0]["lr"]); optimizer.step(); scheduler.step()
print(f"lr: start {_probe[0]:.2e} -> peak {max(_probe):.2e} -> end {_probe[-1]:.2e}")
assert _probe[-1] < max(_probe), "learning rate does not decay -- check min_lr"

# rebuild cleanly after the probe
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate,
                              betas=(0.9, 0.95), weight_decay=0.1, eps=1e-9)
scheduler = SequentialLR(
    optimizer,
    schedulers=[LinearLR(optimizer, total_iters=warmup_steps),
                CosineAnnealingLR(optimizer, T_max=max(total_updates - warmup_steps, 1),
                                  eta_min=min_lr)],
    milestones=[warmup_steps])
scaler = torch.amp.GradScaler(device_type, enabled=(dtype == "float16"))

## Step 8: Pre-train the SLM

**CHANGED: full training state is checkpointed, and re-running this cell
resumes.** The video saves `model.state_dict()` only, which cannot resume —
without the optimizer's momentum the loss jumps when you restart. Colab and
Kaggle *will* disconnect during a run this long, so this matters more than
anything else in the notebook.

`MAX_HOURS` stops and checkpoints before the platform kills the session. If
you get disconnected, just run this cell again.

In [ ]:
import time, math

CKPT = os.path.join(OUT_DIR, "ckpt.pt")
BEST = os.path.join(OUT_DIR, "best_model_params.pt")

start_iter, best_val_loss = 0, float("inf")
history = []

if os.path.exists(CKPT):
    ck = torch.load(CKPT, map_location=device, weights_only=False)
    model.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    scheduler.load_state_dict(ck["scheduler"])
    if ck.get("scaler"): scaler.load_state_dict(ck["scaler"])
    start_iter, best_val_loss = ck["iter"], ck["best_val_loss"]
    history = ck.get("history", [])
    print(f"resumed at iter {start_iter:,}/{max_iters:,}, best val {best_val_loss:.4f}")

def save(path):
    tmp = path + ".tmp"
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict() if scaler.is_enabled() else None,
                "iter": it + 1, "best_val_loss": best_val_loss,
                "history": history, "config": config.__dict__}, tmp)
    os.replace(tmp, path)   # atomic: a kill mid-write keeps the old checkpoint

t0 = time.time()
model.train()
optimizer.zero_grad(set_to_none=True)

for it in tqdm(range(start_iter, max_iters), initial=start_iter, total=max_iters):
    X, y = get_batch("train")
    with ctx:
        logits, loss = model(X, y)
        loss = loss / gradient_accumulation_steps
    scaler.scale(loss).backward()      # outside ctx: backward is not autocast

    if (it + 1) % gradient_accumulation_steps == 0:
        scaler.unscale_(optimizer)     # clip real gradients, not scaled ones
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer); scaler.update()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()               # BUGFIX: per optimizer step

        update = (it + 1) // gradient_accumulation_steps
        if update % EVAL_INTERVAL == 0:
            L = estimate_loss(model)
            gap = L.get("val_seen", float("nan")) - L.get("val_new", float("nan"))
            print(f"\nupdate {update:,}: train {L.get('train', float('nan')):.4f}  "
                  f"val_new {L.get('val_new', float('nan')):.4f}  "
                  f"val_seen {L.get('val_seen', float('nan')):.4f}  gap {gap:+.4f}  "
                  f"lr {optimizer.param_groups[0]['lr']:.2e}")
            history.append({"update": update, **L})
            # Fall back to train loss if val_new is somehow unavailable, so a
            # best checkpoint always exists by the time Step 10 runs.
            metric = L.get("val_new", L.get("train", float("inf")))
            if metric < best_val_loss:
                best_val_loss = metric
                torch.save(model.state_dict(), BEST)
                print(f"  new best {'val_new' if 'val_new' in L else 'train'} "
                      f"{best_val_loss:.4f}")
            save(CKPT)

        if MAX_HOURS and (time.time() - t0) > MAX_HOURS * 3600:
            save(CKPT)
            print(f"\nMAX_HOURS reached at update {update:,}. Checkpointed — "
                  f"re-run this cell to continue.")
            break

save(CKPT)
print(f"\ndone: iter {it+1:,}/{max_iters:,}, {(time.time()-t0)/3600:.2f}h this session, "
      f"best val_new {best_val_loss:.4f}")

## Step 9: Plot the loss

In [ ]:
import matplotlib.pyplot as plt

if history:
    x = [h["update"] for h in history]
    for key, label in (("train", "train"), ("val_new", "val (new templates)"),
                       ("val_seen", "val (seen templates)")):
        ys = [h.get(key) for h in history]
        if any(y is not None for y in ys):
            plt.plot(x, ys, label=label)
    plt.xlabel("optimizer update"); plt.ylabel("cross-entropy loss")
    plt.legend(); plt.grid(alpha=0.3); plt.title("SLM training")
    plt.show()
else:
    print("no history yet — train first")

## Step 10: Generate stories

The video primes with `"Once upon a time there was a pumpkin."` because its
model has no story-boundary token. Yours does, so priming with
`<|endoftext|>` asks for a **completely fresh story** and generation stops
when the model decides the story is over.

The "ended on their own" count is worth watching: a model that never emits
`<|endoftext|>` has not learned how long a story is.

In [ ]:
if os.path.exists(BEST):
    model.load_state_dict(torch.load(BEST, map_location=device))
    print("loaded best_model_params.pt")
elif os.path.exists(CKPT):
    model.load_state_dict(torch.load(CKPT, map_location=device,
                                     weights_only=False)["model"])
    print("no best checkpoint yet -- using the latest one")
else:
    raise SystemExit("No checkpoint found. Run Step 8 first.")
model.eval()

@torch.no_grad()
def write_stories(n=5, max_new_tokens=600, temperature=0.8, top_k=200, prompt=None):
    ids = [eot_id] + (encode(prompt) if prompt else [])
    idx = torch.tensor([ids] * n, dtype=torch.long, device=device)
    done = torch.zeros(n, dtype=torch.bool, device=device)
    for _ in range(max_new_tokens):
        cond = idx[:, -config.block_size:]
        logits, _ = model(cond)
        logits = logits[:, -1, :] / temperature
        if top_k:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("Inf")
        nxt = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
        nxt = torch.where(done.unsqueeze(1), torch.full_like(nxt, eot_id), nxt)
        done |= nxt.squeeze(1) == eot_id
        idx = torch.cat((idx, nxt), dim=1)
        if bool(done.all()): break

    finished = 0
    for row in idx.tolist():
        out = row[1:]
        ended = eot_id in out
        if ended: out = out[:out.index(eot_id)]; finished += 1
        text = decode(out).strip()
        print(f"=== {len(text.split())} words, "
              f"{'complete' if ended else 'TRUNCATED'} ===\n{text}\n")
    print(f"{finished}/{n} ended on their own (training target was 230-260 words)")

write_stories(5)

In [ ]:
# Continue a story you start
write_stories(2, prompt='"You think you can just walk in here and take what is ours?"')

## Save your work before the session dies

Colab and Kaggle wipe the container. Download `best_model_params.pt` and
`tokenizer.json` — you need **both** to generate text later; the checkpoint
alone is unusable without the tokenizer that produced its token ids.

In [ ]:
print(OUT_DIR)
for f in sorted(os.listdir(OUT_DIR)):
    print(f"  {f:28s} {os.path.getsize(os.path.join(OUT_DIR, f))/1e6:8.2f} MB")

# Colab: from google.colab import files; files.download(BEST)
# Kaggle: files in /kaggle/working appear in the notebook Output tab.